In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

print("🚀 Iniciando extracción desde Transfermarkt...")

# -----------------------------
# 🌐 1. CONFIGURACIÓN DE URL Y HEADERS ANTI-BOTS
# -----------------------------
# URL oficial de la Liga Dimayor Apertura en Transfermarkt
url_tm = "https://www.transfermarkt.co/liga-dimayor-apertura/startseite/wettbewerb/COL1"

# Headers pesados para simular tráfico humano desde un navegador real
headers_tm = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "es-ES,es;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Connection": "keep-alive"
}

# -----------------------------
# 📡 2. PETICIÓN HTTP
# -----------------------------
response_tm = requests.get(url_tm, headers=headers_tm)

if response_tm.status_code != 200:
    raise Exception(f"¡Paila! Error de conexión: {response_tm.status_code}. Revisa si Transfermarkt bloqueó la IP.")

# -----------------------------
# 🥣 3. PARSEO DE HTML
# -----------------------------
soup_tm = BeautifulSoup(response_tm.text, "html.parser")

# Buscamos la tabla principal donde están los equipos
tabla_equipos = soup_tm.find("table", class_="items")
filas_tm = tabla_equipos.find("tbody").find_all("tr")

lista_nominas = []

# -----------------------------
# 🔄 4. ITERACIÓN Y EXTRACCIÓN DE FILAS
# -----------------------------
for fila in filas_tm:
    celdas = fila.find_all("td")
    
    # Validamos que la fila tenga datos reales
    if len(celdas) > 3:
        # Extraer nombre del equipo
        equipo_celda = fila.find("td", class_="hauptlink")
        if not equipo_celda:
            continue
            
        equipo_nombre = equipo_celda.text.strip()
        
        # Extraer el valor total de la nómina (última columna)
        valor_raw = celdas[-1].text.strip()
        
        # -----------------------------
        # 💰 5. LIMPIEZA FINANCIERA (CONVERSIÓN A FLOAT)
        # -----------------------------
        valor_limpio = 0.0
        try:
            # Buscar solo los números y la coma (ej: "15,50")
            num_str_match = re.search(r"[\d\,]+", valor_raw)
            if num_str_match:
                # Cambiar coma por punto para que Python lo entienda como float
                num_str = num_str_match.group().replace(",", ".")
                numero = float(num_str)
                
                # Normalizar todo a Millones de Euros
                if "mill" in valor_raw.lower():
                    valor_limpio = numero
                elif "mil" in valor_raw.lower():
                    valor_limpio = numero / 1000  # Pasa miles a millones
        except Exception as e:
            print(f"Error procesando valor de {equipo_nombre}: {e}")
            valor_limpio = 0.0
            
        lista_nominas.append({
            "equipo_tm": equipo_nombre,
            "valor_millones_eur": valor_limpio
        })

# -----------------------------
# 📊 6. CREACIÓN DEL DATAFRAME
# -----------------------------
df_nominas = pd.DataFrame(lista_nominas)

# -----------------------------
# 🔧 7. HOMOLOGACIÓN DE NOMBRES (DICCIONARIO)
# -----------------------------
# Ajustamos los nombres para que hagan match perfecto con tu 'datos.ipynb'
mapa_nombres = {
    "Atlético Nacional": "Atlético Nacional",
    "Millonarios FC": "Millonarios",
    "Junior de Barranquilla": "Junior",
    "América de Cali": "América de Cali",
    "Independiente Santa Fe": "Santa Fe",
    "Independiente Medellín": "Independiente Medellín",
    "Deportes Tolima": "Deportes Tolima",
    "Deportivo Pereira": "Deportivo Pereira",
    "Asociación Deportivo Cali": "Deportivo Cali",
    "Once Caldas": "Once Caldas",
    "Atlético Bucaramanga": "Atlético Bucaramanga",
    "Águilas Doradas": "Águilas Doradas",
    "Asociación Deportivo Pasto": "Deportivo Pasto",
    "Fortaleza CEIF": "Fortaleza",
    "Boyacá Chicó FC": "Boyacá Chicó",
    "Jaguares de Córdoba": "Jaguares",
    "Alianza FC": "Alianza Valledupar",
    "Patriotas Boyacá": "Patriotas",
    "Envigado FC": "Envigado",
    "CD La Equidad Seguros SA": "La Equidad",
    # Agrega o modifica si notas alguna diferencia extra con tu Wikipedia local
}

df_nominas["equipo"] = df_nominas["equipo_tm"].map(mapa_nombres).fillna(df_nominas["equipo_tm"])

# -----------------------------
# 💾 8. EXPORTAR A ARCHIVO LOCAL
# -----------------------------
# Ordenamos las columnas para mayor claridad
df_final_export = df_nominas[["equipo", "equipo_tm", "valor_millones_eur"]]

# Guardamos el archivo CSV (index=False evita que se guarde la columna de los números de fila)
nombre_archivo = "datos_nominas_2026.csv"
df_final_export.to_csv(nombre_archivo, index=False, encoding="utf-8-sig")

print(f"✅ ¡Melo! Scraping completado. Datos guardados en '{nombre_archivo}'.")
display(df_final_export.head())

🚀 Iniciando extracción desde Transfermarkt...
✅ ¡Melo! Scraping completado. Datos guardados en 'datos_nominas_2026.csv'.


,equipo,equipo_tm,valor_millones_eur
0,Atlético Nacional,Atlético Nacional,35.45
1,Millonarios,Millonarios FC,29.93
2,CD América de Cali,CD América de Cali,28.43
3,Junior,Junior de Barranquilla,23.45
4,Independiente Medellín,Independiente Medellín,21.25


In [2]:
display(df_final_export)

,equipo,equipo_tm,valor_millones_eur
0,Atlético Nacional,Atlético Nacional,35.45
1,Millonarios,Millonarios FC,29.93
2,CD América de Cali,CD América de Cali,28.43
3,Junior,Junior de Barranquilla,23.45
4,Independiente Medellín,Independiente Medellín,21.25
5,Santa Fe,Independiente Santa Fe,16.18
6,Deportivo Cali,Deportivo Cali,15.25
7,Águilas Doradas,Águilas Doradas,15.24
8,Internacional de Bogotá,Internacional de Bogotá,14.91
9,Deportes Tolima,Deportes Tolima,14.83


In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
import random

print("🚀 Iniciando Crawler Nivel 2 en Transfermarkt...")

# -----------------------------
# 🌐 1. CONFIGURACIÓN BASE
# -----------------------------
url_base = "https://www.transfermarkt.co"
url_liga = f"{url_base}/liga-dimayor-apertura/startseite/wettbewerb/COL1"

headers_tm = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "es-ES,es;q=0.9",
    "Connection": "keep-alive"
}

# -----------------------------
# 📡 2. PASO 1: EXTRAER LINKS DE LOS EQUIPOS
# -----------------------------
print("Obteniendo URLs de los equipos...")
response_liga = requests.get(url_liga, headers=headers_tm)

if response_liga.status_code != 200:
    raise Exception(f"Bloqueo inicial. Código: {response_liga.status_code}")

soup_liga = BeautifulSoup(response_liga.text, "html.parser")
tabla_equipos = soup_liga.find("table", class_="items").find("tbody").find_all("tr")

equipos_urls = []

for fila in tabla_equipos:
    celdas = fila.find_all("td")
    if len(celdas) > 3:
        link_tag = fila.find("td", class_="hauptlink").find("a")
        if link_tag:
            nombre = link_tag.text.strip()
            # Obtenemos el link parcial (ej: /atletico-nacional/startseite/verein/683)
            href = link_tag.get("href")
            equipos_urls.append({"equipo_tm": nombre, "url": f"{url_base}{href}"})

print(f"✅ ¡Se encontraron {len(equipos_urls)} equipos! Empezando extracción por jugador...\n")

# -----------------------------
# 🕵️‍♂️ 3. PASO 2: NAVEGAR EQUIPO POR EQUIPO (CON CUIDADO)
# -----------------------------
lista_jugadores = []

# OJO: Si quieres probar primero sin raspar los 20 equipos, cambia esto a: equipos_urls[:3]
for equipo in equipos_urls:
    nombre_equipo = equipo["equipo_tm"]
    url_equipo = equipo["url"]
    
    print(f"Camellando con: {nombre_equipo}...")
    
    # Simular comportamiento humano: Pausa aleatoria entre 2 y 5 segundos
    time.sleep(random.uniform(2, 5))
    
    res_equipo = requests.get(url_equipo, headers=headers_tm)
    if res_equipo.status_code != 200:
        print(f"⚠️ Error {res_equipo.status_code} al entrar a {nombre_equipo}. Saltando...")
        continue
        
    soup_equipo = BeautifulSoup(res_equipo.text, "html.parser")
    
    try:
        # En la página del equipo, buscamos la tabla de jugadores
        tabla_jugadores = soup_equipo.find("table", class_="items").find("tbody").find_all("tr", recursive=False)
        
        for fila_jugador in tabla_jugadores:
            # Transfermarkt mete filas ocultas para separar posiciones, las ignoramos
            if "bg_blau_20" in fila_jugador.get("class", []):
                continue
                
            # Extraer nombre del jugador
            tag_nombre = fila_jugador.find("td", class_="hauptlink")
            if not tag_nombre:
                continue
            nombre_jugador = tag_nombre.text.strip()
            
            # El valor del jugador suele estar al final
            tag_valor = fila_jugador.find_all("td", class_="rechts")[-1]
            valor_raw = tag_valor.text.strip()
            
            # --- Limpieza numérica ---
            valor_limpio = 0.0
            if valor_raw != "-" and valor_raw != "":
                try:
                    num_str_match = re.search(r"[\d\,]+", valor_raw)
                    if num_str_match:
                        num_str = num_str_match.group().replace(",", ".")
                        numero = float(num_str)
                        if "mill" in valor_raw.lower():
                            valor_limpio = numero
                        elif "mil" in valor_raw.lower():
                            valor_limpio = numero / 1000
                except:
                    valor_limpio = 0.0
            
            lista_jugadores.append({
                "equipo_tm": nombre_equipo,
                "jugador": nombre_jugador,
                "valor_millones_eur": valor_limpio
            })
            
    except AttributeError:
        print(f"⚠️ No se encontró la tabla de jugadores para {nombre_equipo}")

# -----------------------------
# 📊 4. CREACIÓN Y EXPORTACIÓN DEL DATAFRAME
# -----------------------------
df_jugadores = pd.DataFrame(lista_jugadores)

# Guardamos el archivo
nombre_archivo = "datos_jugadores_2026.csv"
df_jugadores.to_csv(nombre_archivo, index=False, encoding="utf-8-sig")

print(f"\n✅ ¡Melo total! Se extrajeron {len(df_jugadores)} jugadores en total.")
print(f"Archivo guardado como '{nombre_archivo}'.")

display(df_jugadores.head(10))

🚀 Iniciando Crawler Nivel 2 en Transfermarkt...
Obteniendo URLs de los equipos...
✅ ¡Se encontraron 20 equipos! Empezando extracción por jugador...

Camellando con: Atlético Nacional...
Camellando con: Millonarios FC...
Camellando con: CD América de Cali...
Camellando con: Junior de Barranquilla...
Camellando con: Independiente Medellín...
Camellando con: Independiente Santa Fe...
Camellando con: Deportivo Cali...
Camellando con: Águilas Doradas...
Camellando con: Internacional de Bogotá...
Camellando con: Deportes Tolima...
Camellando con: Once Caldas...
Camellando con: Atlético Bucaramanga...
Camellando con: Deportivo Pereira...
Camellando con: Llaneros FC...
Camellando con: Unión Magdalena...
Camellando con: Asociación Deportivo Pasto...
Camellando con: Alianza FC...
Camellando con: Envigado FC...
Camellando con: Boyacá Chicó FC...
Camellando con: Fortaleza CEIF...

✅ ¡Melo total! Se extrajeron 1007 jugadores en total.
Archivo guardado como 'datos_jugadores_2026.csv'.


,equipo_tm,jugador,valor_millones_eur
0,Atlético Nacional,David Ospina,2.50
1,Atlético Nacional,Harlen Castillo,0.50
2,Atlético Nacional,Luis Marquinez,0.45
3,Atlético Nacional,Edimer Zea,0.00
4,Atlético Nacional,César Haydar,1.00
5,Atlético Nacional,Juan Felipe Aguirre,0.90
6,Atlético Nacional,William Tesillo,0.90
7,Atlético Nacional,Juan Jose Arias,0.35
8,Atlético Nacional,Simón García,0.00
9,Atlético Nacional,Royer Caicedo,0.00
